# DP1 inner-bin postage stamp blend review

This notebook inspects the `dp1_default` foreground-background pairs in the 10-15 kpc projected radial bin. It builds or loads the local pair table, checks that the expected 111 pairs are present, then uses Rubin Science Platform TAP/SIA/SODA services to retrieve DP1 deep-coadd cutouts and write a contact-sheet review.

Run the image-access cells inside the Rubin Science Platform Notebook Aspect at `data.lsst.cloud`.

In [ ]:
from pathlib import Path
import io
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy import units as u
from astropy.io import fits
from astropy.wcs import WCS

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if (ROOT / "src").exists() and str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from dusty_colors.postage_stamps import (
    DEFAULT_EXPECTED_INNER_PAIR_COUNT,
    available_tap_columns,
    build_projected_pair_table,
    fetch_object_metadata,
    pair_table_summary,
    requested_blend_columns,
    score_pair_blend_risk,
    select_existing_columns,
    write_contact_sheet_html,
    write_review_labels_template,
)

SAMPLE_DIR = ROOT / "results" / "samples" / "dp1_default"
OUT_DIR = ROOT / "results" / "postage_stamps" / "dp1_default"
STAMP_DIR = OUT_DIR / "stamps"
PAIR_TABLE_PATH = OUT_DIR / "inner_bin_pairs.csv"

BANDS = ("r", "i")
STAMP_FOV_ARCSEC = 12.0
OVERVIEW_FOV_ARCSEC = 20.0
GENERATE_OVERVIEWS_FOR_HIGH_RISK = True
RUN_ALL_PAIRS = False
PILOT_COUNT = 5

OUT_DIR.mkdir(parents=True, exist_ok=True)
STAMP_DIR.mkdir(parents=True, exist_ok=True)

## Build or load the local pair table

If `inner_bin_pairs.csv` already exists, this cell loads it. Otherwise it rebuilds the table from the prepared foreground/background parquet files using the same projected-separation definition as the stacker.

In [ ]:
if PAIR_TABLE_PATH.exists():
    pairs = pd.read_csv(PAIR_TABLE_PATH)
else:
    foreground = pd.read_parquet(
        SAMPLE_DIR / "foreground.parquet",
        columns=["object_id", "ra", "dec", "z_phot", "field"],
    )
    background = pd.read_parquet(
        SAMPLE_DIR / "background.parquet",
        columns=["object_id", "ra", "dec", "z_phot", "field"],
    )
    pairs = build_projected_pair_table(foreground, background)
    pairs.to_csv(PAIR_TABLE_PATH, index=False)

summary = pair_table_summary(pairs)
assert len(pairs) == DEFAULT_EXPECTED_INNER_PAIR_COUNT, summary
print(f"pair count: {summary['count']}")
display(pd.DataFrame([
    {"metric": "theta_arcsec", **summary["theta_arcsec"]},
    {"metric": "r_perp_kpc", **summary["r_perp_kpc"]},
]))
display(pairs.head())

## Connect to RSP services

The remaining cells require authenticated DP1 access through the Rubin Science Platform.

In [ ]:
try:
    from lsst.rsp import get_tap_service
    from lsst.rsp.service import get_siav2_service
    from lsst.rsp.utils import get_pyvo_auth
    from pyvo.dal.adhoc import DatalinkResults, SodaQuery
except ImportError as exc:
    raise RuntimeError(
        "Run the image-access cells inside the Rubin Science Platform Notebook Aspect."
    ) from exc

tap_service = get_tap_service("tap")
sia_service = get_siav2_service("dp1")
session = get_pyvo_auth()
assert tap_service is not None
assert sia_service is not None

## Fetch Object-table blend metadata

In [ ]:
available_columns = available_tap_columns(tap_service)
requested_columns = requested_blend_columns(BANDS)
object_columns = select_existing_columns(requested_columns, available_columns)
missing_columns = sorted(set(requested_columns) - set(object_columns))
print("Using Object columns:", object_columns)
print("Missing requested columns:", missing_columns)

object_ids = pd.unique(
    pd.concat([pairs["foreground_object_id"], pairs["background_object_id"]])
    .dropna()
    .astype("int64")
)
object_metadata = fetch_object_metadata(tap_service, object_ids, object_columns)
object_metadata.to_csv(OUT_DIR / "object_blend_metadata.csv", index=False)

review = score_pair_blend_risk(pairs, object_metadata, bands=BANDS)
review.to_csv(OUT_DIR / "pair_blend_review_table.csv", index=False)
write_review_labels_template(review, OUT_DIR / "review_labels.csv")
display(review[["pair_id", "theta_arcsec", "blend_risk", "same_parent", "max_blendedness", "max_overlap_fraction"]].head(10))

## Cutout and plotting helpers

In [ ]:
def find_deep_coadd_access_url(ra, dec, band, search_radius_deg=0.001):
    results = sia_service.search(pos=(ra, dec, search_radius_deg), calib_level=3)
    table = results.to_table()
    if len(table) == 0:
        return None
    df = table.to_pandas()
    mask = pd.Series(True, index=df.index)
    if "lsst_band" in df:
        mask &= df["lsst_band"].astype(str) == band
    if "dataproduct_subtype" in df:
        subtype = df["dataproduct_subtype"].astype(str)
        deep = subtype.str.contains("deep_coadd", case=False, na=False)
        deep |= subtype.str.contains("deepCoadd", case=False, na=False)
        mask &= deep
    candidates = df.loc[mask]
    if candidates.empty:
        return None
    return str(candidates.iloc[0]["access_url"])


def get_cutout_hdul(access_url, ra, dec, fov_arcsec):
    dl_result = DatalinkResults.from_result_url(access_url, session=session)
    query = SodaQuery.from_resource(
        dl_result,
        dl_result.get_adhocservice_by_id("cutout-sync"),
        session=session,
    )
    fov_deg = fov_arcsec / 3600.0
    query.circle = (ra * u.deg, dec * u.deg, 0.5 * fov_deg * u.deg)
    payload = query.execute_stream().read()
    query.raise_if_error()
    return fits.open(io.BytesIO(payload))


def science_hdu(hdul):
    for hdu in hdul:
        if getattr(hdu, "data", None) is not None:
            return hdu
    raise ValueError("Cutout returned no image HDU")


def plot_pair_stamps(row, *, fov_arcsec=STAMP_FOV_ARCSEC, suffix="stamps"):
    pair_id = row["pair_id"]
    hdus = {}
    status = {"pair_id": pair_id, "suffix": suffix, "status": "ok", "message": ""}
    for band in BANDS:
        try:
            access_url = find_deep_coadd_access_url(row["midpoint_ra"], row["midpoint_dec"], band)
            if access_url is None:
                status["status"] = "missing_image"
                status["message"] += f"missing {band}; "
                continue
            hdus[band] = science_hdu(
                get_cutout_hdul(access_url, row["midpoint_ra"], row["midpoint_dec"], fov_arcsec)
            )
        except Exception as exc:
            status["status"] = "cutout_error"
            status["message"] += f"{band}: {exc}; "

    if not hdus:
        return status

    fig = plt.figure(figsize=(4.2 * len(hdus), 4.2), constrained_layout=True)
    for index, (band, hdu) in enumerate(hdus.items(), start=1):
        wcs = WCS(hdu.header)
        ax = fig.add_subplot(1, len(hdus), index, projection=wcs)
        data = np.asarray(hdu.data, dtype=float)
        finite = data[np.isfinite(data)]
        vmin, vmax = np.nanpercentile(finite, [1, 99]) if finite.size else (None, None)
        ax.imshow(data, origin="lower", cmap="gray", vmin=vmin, vmax=vmax)
        world = ax.get_transform("world")
        ax.scatter(row["foreground_ra"], row["foreground_dec"], transform=world, marker="o", facecolors="none", edgecolors="tab:red", s=95, linewidths=1.8, label="fg")
        ax.scatter(row["background_ra"], row["background_dec"], transform=world, marker="+", color="tab:cyan", s=115, linewidths=2.0, label="bg")
        ax.plot([row["foreground_ra"], row["background_ra"]], [row["foreground_dec"], row["background_dec"]], transform=world, color="tab:yellow", linewidth=1.0)
        ax.set_title(f"{band} band")
        ax.set_xlabel("RA")
        ax.set_ylabel("Dec")
    fig.suptitle(
        f"{pair_id}  theta={row['theta_arcsec']:.3f} arcsec  "
        f"z_fg={row['foreground_z']:.3f} z_bg={row['background_z']:.3f}  "
        f"risk={row.get('blend_risk', 'unknown')}"
    )
    output = STAMP_DIR / f"{pair_id}_{suffix}.png"
    fig.savefig(output, dpi=150)
    plt.close(fig)
    status["output"] = str(output)
    return status

## Generate pilot or full stamp set

The default pilot mode selects up to five pairs, preferring different fields and then filling by smallest separation. Set `RUN_ALL_PAIRS = True` in the configuration cell to render every pair.

In [ ]:
def select_pilot_pairs(table, count=PILOT_COUNT):
    selected = []
    for _, group in table.sort_values("theta_arcsec").groupby("foreground_field", dropna=False):
        selected.append(group.iloc[0])
        if len(selected) >= count:
            break
    if len(selected) < count:
        used = {row["pair_id"] for row in selected}
        for _, row in table.sort_values("theta_arcsec").iterrows():
            if row["pair_id"] in used:
                continue
            selected.append(row)
            if len(selected) >= count:
                break
    return pd.DataFrame(selected)


pairs_to_render = review if RUN_ALL_PAIRS else select_pilot_pairs(review)
print(f"Rendering {len(pairs_to_render)} of {len(review)} pairs")

statuses = []
for _, row in pairs_to_render.iterrows():
    statuses.append(plot_pair_stamps(row, fov_arcsec=STAMP_FOV_ARCSEC, suffix="stamps"))
    if GENERATE_OVERVIEWS_FOR_HIGH_RISK and row.get("blend_risk") == "high":
        statuses.append(plot_pair_stamps(row, fov_arcsec=OVERVIEW_FOV_ARCSEC, suffix="overview"))

status_table = pd.DataFrame(statuses)
status_table.to_csv(OUT_DIR / "stamp_status.csv", index=False)
display(status_table)

## Write the contact sheet

In [ ]:
rendered_review = review[review["pair_id"].isin(pairs_to_render["pair_id"])].copy()
report_path = write_contact_sheet_html(
    rendered_review,
    OUT_DIR / "contact_sheet.html",
    image_dir=STAMP_DIR,
    image_template="{pair_id}_stamps.png",
)
print(report_path)
print(OUT_DIR / "review_labels.csv")